# Model: the entity classes

The model is the set of Python classes in `lode.models` that describe what LODE finds in a document. The [reader](reader.ipynb) creates them, the [artefact](artefact.ipynb) holds them and the viewer reads them. They are plain Python: no RDF parsing and no rendering.

Every class follows the same rules: one setter and one getter per field, multi-valued fields are lists, and the inverse of a link is set for you. This notebook inspects the classes and shows each rule on a small ontology.

:::{warning} Beta version · unstable
LODE 2 is under development and in beta: anything described in this documentation may change. This page describes internals that can change without notice.
:::

## Methods to inspect the model

| Method | What it gives |
|---|---|
| `lode.models.__all__` | The names of the classes of the model. |
| `cls.__subclasses__()` | The children of a class: the hierarchy. |
| `inspect.getdoc(cls)` | What a class is, from its docstring. |
| `vars(cls())` | The fields of a class, with their empty values. |
| `entity.get_<field>()` | The getter of a field. |
| `dir(cls)` | The setters of a class (`set_<field>`), which the mapper names. |
| `repr(entity)` | The class and the identifier of an entity. |

## Examples

One for each way of the table.

### `lode.models.__all__`

In [1]:
import lode.models as models

print(len(models.__all__), "classes")
print(models.__all__)

25 classes
['Resource', 'Property', 'Relation', 'Annotation', 'Attribute', 'Literal', 'Concept', 'Datatype', 'Restriction', 'TruthFunction', 'OneOf', 'PropertyConceptRestriction', 'Value', 'Quantifier', 'Cardinality', 'Model', 'Individual', 'Statement', 'Container', 'Collection', 'DatatypeRestriction', 'Variable', 'Atom', 'Rule', 'PropertySelfRestriction']


### `cls.__subclasses__()` and `inspect.getdoc(cls)`

Almost everything extends `Resource`. `Literal` does not: it is a value, not a resource. The tree is built from the classes, so it cannot go out of date.

In [2]:
import inspect

from lode.models import Literal, Resource


def show_tree(cls, depth=0):
    """Print a class and its subclasses, each with the first line of its docstring."""
    summary = (inspect.getdoc(cls) or "").split("\n")[0]
    print(f"{'  ' * depth}{cls.__name__:<{30 - 2 * depth}} {summary}")
    for sub in cls.__subclasses__():
        show_tree(sub, depth + 1)


show_tree(Resource)
print()
print("Literal is a Resource:", issubclass(Literal, Resource))

Resource                       An RDF resource, with its annotations (labels, notes, ...) and links to other resources.
  Property                     Represents an RDF Property
    Relation                   Represents an RDF Object Property
    Annotation                 An annotation property: it carries documentation, not logical meaning.
    Attribute                  A datatype property: it relates a resource to literals.
  Concept                      An OWL class or a SKOS concept.
    Datatype                   A datatype: the kind of value a literal can have.
    Restriction                Base of the anonymous classes built by restrictions and logical operators.
      TruthFunction            A class expression combining concepts with a logical operator (and, or, not).
      OneOf                    A class defined by listing its members (owl:oneOf).
      PropertyConceptRestriction Base of the restrictions that apply a property to a concept (quantifiers, cardinalities).
   

### `vars(cls())`

The fields of a class. Cardinalities are written as comments next to each field in the class.

In [3]:
from lode.models import Concept

print(len(vars(Concept())), "fields")
print(list(vars(Concept())))

36 fields
['has_identifier', 'is_deprecated', 'has_comment', 'has_label', 'has_preferred_label', 'has_alternative_label', 'has_hidden_label', 'has_notation', 'has_note', 'has_change_note', 'has_definition', 'has_editorial_note', 'has_example', 'has_history_note', 'has_scope_note', 'see_also', 'is_defined_by', 'has_version_info', 'also_defined_as', 'has_type', 'is_included_in', 'is_in_range_of', 'is_sub_concept_of', 'is_super_concept_of', 'is_disjoint_with', 'is_equivalent_to', 'is_related_to', 'has_broad_match', 'has_narrow_match', 'has_related_match', 'has_exact_match', 'has_close_match', 'is_top_concept_of', 'has_key', 'individuals_with_this_type', 'is_in_domain_of']


### `entity.get_<field>()`

The entities below come from a small ontology. A getter returns a copy of a list, so callers cannot alter the model by accident.

In [4]:
import tempfile
from pathlib import Path

from lode import OwlOntology

TURTLE = """
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ex: <http://example.org/zoo#> .

<http://example.org/zoo> a owl:Ontology .
ex:Animal a owl:Class ; rdfs:label "Animal"@en .
ex:Cat a owl:Class ; rdfs:label "Cat"@en ; rdfs:subClassOf ex:Animal .
"""

path = Path(tempfile.mkdtemp()) / "zoo.ttl"
path.write_text(TURTLE, encoding="utf-8")
artefact = OwlOntology(str(path))

cat = next(iter(artefact.get_entities_by_iri("http://example.org/zoo#Cat")))
animal = next(iter(artefact.get_entities_by_iri("http://example.org/zoo#Animal")))

print([label.get_has_value() for label in cat.get_has_label()])
print(cat.get_is_sub_concept_of())

['Cat']
[Concept('http://example.org/zoo#Animal')]


### The setters, with `dir(cls)`

The reader fills the model only through the setters, named in the YAML mapper. Setting a link also sets its inverse:

In [5]:
setters = [name for name in dir(Concept) if name.startswith("set_")]
print(len(setters), "setters, for example", setters[:3])

print(animal.get_is_super_concept_of())

36 setters, for example ['set_also_defined_as', 'set_has_alternative_label', 'set_has_broad_match']
[Concept('http://example.org/zoo#Cat')]


### `repr(entity)`

The class and the identifier, never the relations, which would recurse.

In [6]:
print(repr(cat))

Concept('http://example.org/zoo#Cat')


## How LODE uses it

The reader creates the entities as instances of these classes and stores them in the artefact. The viewer reads them to build what the templates render.

## Writing your own model classes

This page only inspects the classes that ship with LODE. A profile that needs its own can name them in its `reader.yaml` as `module:Name`, with their setters and getters. To write it, go to [Extending LODE, the mapper](../extending.md#step-1-write-the-mapper).